In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain.agents.middleware import dynamic_prompt, ModelRequest

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

from langchain.agents import create_agent

from dotenv import load_dotenv

c:\Users\USER\RAG-agent-with-LangChain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\USER\AppData\Local\Temp\ipykernel_7556\2988364427.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader


In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    vertexai=True,
    temperature=0,
    )

In [4]:
loader = DirectoryLoader('../data', 
                         glob="*.pdf", 
                         loader_cls=PyPDFLoader
)

In [5]:
docs = loader.load()

In [6]:
docs

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-05T13:40:36+00:00', 'author': 'USER', 'moddate': '2026-05-05T13:40:37+00:00', 'source': '..\\data\\Ezekiel_Oluyale_Resume.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='OLUYALE EZEKIEL OREOLUWA \n  Address: Plot E/B Awolowo Street, Oke-Ado, Ibadan, Oyo State, Nigeria \n     Mobile: (+234) 814 829 9173 \n              Email: ezekieloluyale@gmail.com \n      Github: https://github.com/EzekielOluyale \n        Portfolio: https://ezekiel-oluyale-portfolio.streamlit.app \n \nEDUCATION          _      \nFederal University Oye-Ekiti, FUOYE, Ekiti State, Nigeria                         July 2025 \nBEng in Computer Engineering:  \nFirst Class. CGPA: 4.51/5.00 (90.2%) \n \nWORK EXPERIENCE          \namusEcode, Nigeria (Remote)         2019 - Present \nTitle: Founder           \n\uf0b7 Design, build, and deploy scalable ML models that power real-world applications.

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  
    chunk_overlap=100,  
    add_start_index=True,  
    separators=["\n\n", "\n", " ", ""]
)

In [8]:
all_splits = text_splitter.split_documents(docs)

In [9]:
all_splits

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-05T13:40:36+00:00', 'author': 'USER', 'moddate': '2026-05-05T13:40:37+00:00', 'source': '..\\data\\Ezekiel_Oluyale_Resume.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'start_index': 0}, page_content='OLUYALE EZEKIEL OREOLUWA \n  Address: Plot E/B Awolowo Street, Oke-Ado, Ibadan, Oyo State, Nigeria \n     Mobile: (+234) 814 829 9173 \n              Email: ezekieloluyale@gmail.com \n      Github: https://github.com/EzekielOluyale \n        Portfolio: https://ezekiel-oluyale-portfolio.streamlit.app \n \nEDUCATION          _      \nFederal University Oye-Ekiti, FUOYE, Ekiti State, Nigeria                         July 2025 \nBEng in Computer Engineering:  \nFirst Class. CGPA: 4.51/5.00 (90.2%)'),
 Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-05T13:40:36+00:00', 'author': 'USER', 'moddate': '2026-05-05T13:40

In [10]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="text-embedding-004",
    vertexai=True
)

In [11]:
embeddings

GoogleGenerativeAIEmbeddings(client=<google.genai.client.Client object at 0x000001BB9B9F1950>, model='text-embedding-004', task_type=None, google_api_key=SecretStr('**********'), credentials=None, vertexai=True, project=None, location=None, base_url=None, additional_headers=None, client_args=None, api_version=None, request_options=None, output_dimensionality=None)

In [12]:
pc = Pinecone()

index = pc.Index('rag')

In [13]:
index.delete(delete_all=True)

{}

In [14]:
vector_store = PineconeVectorStore(embedding=embeddings, index=index)

In [16]:
vector_store

In [15]:
document_ids = vector_store.add_documents(documents=all_splits)

In [17]:
document_ids

['62bb1c10-675c-4deb-9fe0-d4cd82d61659',
 'ecae8912-7e33-41b4-9961-87f3b4fd922e',
 '2ca4015e-0fbc-42eb-a7cc-367916e688b8',
 'b75554d7-cdf7-42d9-9f1a-dfb022a787b7',
 'be645b98-4dba-4555-be0a-90736c3b58ff',
 '1c5a2510-0a0a-4b99-8816-fbdab1a1ff61',
 'ba349714-0b0f-4afe-8412-78365d4d08af',
 '7df195f4-4bb2-461c-9196-8b102610933e',
 'bcfd6dcb-7075-4e29-8a25-7ba4528d13cc',
 '25eab504-0a11-49df-a3ad-eb15e5b04b49',
 '1bc82d8f-0c51-477b-8a1d-ea0667457801',
 '99ccb1e6-065b-4a8d-bfdd-0dd3c3b5ff01',
 '499c5024-5425-477e-871a-65b3efd04e5d',
 '5269ca79-da4f-4bd6-a50d-15e1a3c6d180',
 'fedc90ca-0ea1-473a-b106-913b69270c66',
 '26015db4-6733-4857-8b94-77c89296ecaa',
 '7df1c5d1-412d-48ef-8205-66ee15364103',
 '7b8bb70d-a8ce-43cf-bae3-ccab842549a5',
 'da9d22fd-4f8c-4b9b-9501-e608b7cfa19c',
 '15659a85-1c75-4033-930e-40d7a2457983',
 '181324e3-661a-4a22-a050-388d65eebb00',
 '08634a56-f9af-4e06-9ced-cc9cf6aa2d0a']